# IAC-Flow — ToothFairy3 Left/Right IAC — Colab uçtan uca çalıştırıcı (v1.2)

Bu notebook **README'deki 8 adımlık boru hattının tamamını**, leakage-free olacak
şekilde sırayla çalıştırır. Yukarıdan aşağıya, hücreleri sırayla çalıştır.

## v1.2'de düzeltilenler (bir önceki koşuda bulunan gerçek hatalar)

1. **Kaynak veri LPS geldiğinde artık düşürülmüyor** — affine'in gerçek yön kosinüslerine
   göre RPI'ye çevriliyor (fiziksel sol/sağ korunur, rastgele flip değil).
2. **S (held-out external test) artık nnU-Net eğitim verisine hiç girmiyor.**
   `prepare_iac_dataset.py --subset PF` ile Dataset801'in ham klasörü SADECE P+F
   içeriyor; S ayrıca, sadece en sonda, ayrı bir klasöre hazırlanıyor.
3. **`configs/splits.json`, nnU-Net'in GERÇEKTEN kullandığı split'e yazılıyor.**
   Yeni `nnunet/write_nnunet_splits.py`, bizim P/F stratified fold'larımızı
   `nnUNet_preprocessed/.../splits_final.json`'a kurar — bu olmadan nnU-Net kendi
   rastgele split'ini üretir ve `predict_oof.py`'ın "held-out" sandığı vakalar aslında
   o modelin eğitiminde görülmüş olabilirdi (leakage).
4. **FAST modu artık dataset ile split'i aynı vaka listesinden üretiyor** (stratified
   P+F örneklemi, `outputs/fast_ids.json`) — eskiden `--limit 40` alfabetik ilk 40'ı
   alıyordu ki bu her zaman sadece 'F' harfiyle başlayanları seçiyordu (F < P < S).
5. **nnU-Net eğitimi artık `subprocess.run(check=True)`** — hata sessizce yutulup
   sonraki fold'a geçilmiyor; `--c` sadece gerçekten bir `checkpoint_latest.pth`
   varsa ekleniyor (ilk çalıştırmada koşulsuz `--c` bazı sürümlerde hataya yol açabilir).

> Not: bir dış inceleme ayrıca "480 = ToothFairy2'den kalma" dedi — bu **yanlış**;
> 480 = P(417)+F(63), gerçek bir çalıştırmada doğrulandı (`create_folds.py` çıktısı).
> Ve `REPO_URL` placeholder'ı bir hata değil, senin doldurman gereken tek satır.

## Önce 3 şeyi bil

1. **Kod repoda, notebook sadece 'çağırıcı'.** İş `data/`, `nnunet/`, `flow/` altında.
2. **Eğitim için GPU şart** (Runtime → Change runtime type → GPU). CPU sadece self-test için yeter.
3. **Saatler–günler sürer.** Colab koparsa aynı hücreyi tekrar çalıştır — checkpoint'ler
   Drive'da, nnU-Net `--c` (varsa) ve flow `--resume` ile kaldığı yerden devam eder.

> **İlk denemede `FAST = True` bırak.** 50-epoch trainer + küçük stratified örneklemle
> tüm zincirin uçtan uca **doğru** çalıştığını ~1 günde görürsün. Sonra `FAST = False`.

## 0. GPU kontrol + Drive bağla + repo kodunu getir
Kod her oturumda runtime'a gelmeli (Drive'a değil).

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — Runtime>GPU sec!")
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO_URL = "https://github.com/ColdVI/ToothFairy3-IAC-Segmentation-Flow.git"  # <-- DUZENLE (gerekirse)
REPO = "/content/ToothFairy3-IAC-Segmentation-Flow"
if not os.path.isdir(REPO):
    os.system(f"git clone {REPO_URL} {REPO}")
    # VEYA repoyu Drive'a attiysan (ustteki satiri yorumla):
    # import shutil; shutil.copytree('/content/drive/MyDrive/ToothFairy3-IAC-Segmentation-Flow', REPO)
assert os.path.isdir(REPO), f"repo yok: {REPO} — REPO_URL'i duzelt veya Drive kopyasini kullan"
os.chdir(REPO)
print("cwd:", os.getcwd())
print("icerik:", sorted(os.listdir(REPO))[:12])

## 0.5 Dataset'i bul (ToothFairy3)

Şu öncelikle bulur: (1) Drive'da zaten açık klasör → `DRIVE_DIR` (senin durumun, zipsiz,
doğrudan okunur, kopyalanmaz), (2) daha önce runtime'a açılmış kopya, (3) sadece
`ToothFairy3.zip` varsa açar. `DRIVE_DIR`'i kendi Drive yoluna göre düzelt.

In [ ]:
import os, zipfile, glob
DRIVE_DIR  = "/content/drive/MyDrive/ToothFairy/ToothFairy3/ToothFairy3"  # <-- DUZENLE
ZIP_PATH   = "/content/drive/MyDrive/ToothFairy3.zip"   # (yalnizca zip halinde varsa)
EXTRACT_TO = "/content/work/tf3_src"

def find_tf3_root(base):
    if not base or not os.path.isdir(base):
        return None
    cand = [base] + [os.path.dirname(dj) for dj in
                     glob.glob(os.path.join(base, "**", "dataset.json"), recursive=True)]
    for r in cand:
        if (os.path.isdir(os.path.join(r, "imagesTr")) and os.path.isdir(os.path.join(r, "labelsTr"))
                and os.path.isfile(os.path.join(r, "dataset.json"))):
            return r
    return None

root = find_tf3_root(DRIVE_DIR)
if root:
    print("Drive'daki acik klasor kullaniliyor (kopyalanmadan):", root)
else:
    root = find_tf3_root(EXTRACT_TO)
    if root is None:
        assert os.path.isfile(ZIP_PATH), (
            f"Dataset bulunamadi.\n  Acik klasor beklenen: {DRIVE_DIR}\n  Zip beklenen: {ZIP_PATH}\n"
            "DRIVE_DIR'i Drive'daki gercek yola ayarla.")
        os.makedirs(EXTRACT_TO, exist_ok=True)
        print("zip aciliyor...", flush=True)
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall(EXTRACT_TO)
        root = find_tf3_root(EXTRACT_TO)

assert root, "imagesTr/labelsTr/dataset.json bulunamadi — DRIVE_DIR'i kontrol et."
print("TF3 kok ->", root)
print("  imagesTr:", len(os.listdir(os.path.join(root, 'imagesTr'))),
      "| labelsTr:", len(os.listdir(os.path.join(root, 'labelsTr'))))

## 1. Ayarlar — burayı düzenle, gerisi türetilir
`FAST=True`: küçük stratified örneklem + 50-epoch trainer (ilk uçtan uca doğrulama).
`FAST=False`: tam P+F (480 vaka) + tam eğitim.

In [ ]:
import os
TF3_SRC    = root                                  # 0.5 hucresinde bulunan yol
DRIVE_RUNS = "/content/drive/MyDrive/ToothFairy/ToothFairy3/iac_runs"  # checkpoint/sonuclar (kalici)

# --- KOSU MODU ---
FAST = False           # True: 40 vaka + 50 epoch, sadece pipeline dogrulama (hizli deneme)
                       # False: tam development seti (~480) ile gercek egitim
FULL_EPOCHS = 1000     # FAST=False iken epoch butcesi -> 250 (hizli/kaynak-dostu) veya 1000 (nnU-Net default, en iyi)
FAST_N = 40            # FAST modda kac development vakasi kullanilsin

WORK = "/content/work"
os.environ["nnUNet_raw"]          = f"{WORK}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{WORK}/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = f"{DRIVE_RUNS}/nnUNet_results"   # Drive: oturum kopsa da kalir
for d in ["nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"]:
    os.makedirs(os.environ[d], exist_ok=True)
os.makedirs(DRIVE_RUNS, exist_ok=True)
os.makedirs("outputs", exist_ok=True)

DST     = f"{os.environ['nnUNet_raw']}/Dataset801_IAC_LR"   # SADECE development (P+F); S girmez
IMAGES  = f"{DST}/imagesTr"
LABELS  = f"{DST}/labelsTr"

# Trainer secimi: FAST -> 50ep; degilse FULL_EPOCHS'a gore 250ep ya da 1000ep (base = nnU-Net default)
if FAST:
    TRAINER = "nnUNetTrainerIAC_NoMirror_50ep"
elif FULL_EPOCHS == 250:
    TRAINER = "nnUNetTrainerIAC_NoMirror_250ep"
elif FULL_EPOCHS == 1000:
    TRAINER = "nnUNetTrainerIAC_NoMirror"          # base trainer = 1000 epoch (nnU-Net default)
else:
    raise ValueError(f"FULL_EPOCHS 250 veya 1000 olmali (verilen: {FULL_EPOCHS})")

assert os.path.isdir(TF3_SRC), f"bulunamadi: {TF3_SRC} — 0.5 hucresini calistirdin mi?"
print("TF3_SRC =", TF3_SRC)
print(f"FAST = {FAST} | FULL_EPOCHS = {FULL_EPOCHS} | trainer = {TRAINER}")
print("nnUNet_results ->", os.environ["nnUNet_results"])

## 2. Bağımlılıklar

In [ ]:
!pip -q install nnunetv2 nibabel scipy scikit-image pyyaml

## 3. (Opsiyonel ama önerilir) CPU self-test — zinciri kurmadan mekanizmayı doğrula
Saniyeler sürer, GPU/veri gerektirmez. Yeşil geçerse residual-flow makinesi sağlam.

In [ ]:
!python flow/selftest.py

---
# ADIM 1 — split üret → `configs/splits.json` (ÖNCE, tüm dataset üzerinden)
P+F = development (5-fold CV), S = held-out external OOD test — **bu her zaman TAM
TF3_SRC üzerinden, tam ve doğru olarak** üretilir (ucuz bir adım, sadece dosya listesi).

`FAST=True` iken ayrıca development'tan **stratified küçük bir örneklem** (`outputs/fast_ids.json`)
seçilir ve o TAM AYNI id listesi hem dataset hazırlama hem split üretiminde kullanılır —
böylece nnU-Net'in gördüğü vakalarla split'in tanımladığı vakalar birebir örtüşür.

In [ ]:
!python data/create_folds.py --src "$TF3_SRC" --out configs/splits.json
import json
s = json.load(open("configs/splits.json"))
print("folds:", len(s["folds"]), "| external S:", len(s["external_test"]), "| dev:", len(s["development"]))

SPLITS = "configs/splits.json"
if FAST:
    import re, random
    dev_ids = s["development"]
    p_ids = [c for c in dev_ids if re.match(r"ToothFairy3P_", c)]
    f_ids = [c for c in dev_ids if re.match(r"ToothFairy3F_", c)]
    rng = random.Random(1234)
    rng.shuffle(p_ids); rng.shuffle(f_ids)
    n_f = min(len(f_ids), max(1, round(FAST_N * len(f_ids) / len(dev_ids))))
    n_p = FAST_N - n_f
    fast_ids = sorted(p_ids[:n_p] + f_ids[:n_f])
    json.dump(fast_ids, open("outputs/fast_ids.json", "w"))
    print(f"[FAST] stratified orneklem: {len(fast_ids)} vaka ({n_p} P + {n_f} F) -> outputs/fast_ids.json")

    !python data/create_folds.py --src "$TF3_SRC" --ids-file outputs/fast_ids.json --out configs/splits_fast.json
    SPLITS = "configs/splits_fast.json"
    sf = json.load(open(SPLITS))
    print("[FAST] splits_fast.json folds:", len(sf["folds"]), "| dev:", len(sf["development"]))

print("\n>>> Kullanilacak split dosyasi (SPLITS):", SPLITS)

## 1.5 Preprocess cache (Colab çok sık kopuyorsa BUNU kullan)
`nnUNetv2_plan_and_preprocess` deterministiktir (rastgelelik yok) — ama çıktısı
`WORK="/content/work"` altında, yani runtime her koptuğunda siliniyor ve baştan
bekliyorsun. Çözüm: ADIM 2 + A1 (preprocess) + A3 (split kurulumu) bittikten sonra
`nnUNet_raw/Dataset801_IAC_LR` + `nnUNet_preprocessed/Dataset801_IAC_LR` tek bir
tar dosyası olarak Drive'a yazılır (aşağıdaki hücre, en sonda). Bir sonraki oturumda
bu hücre o tar'ı bulur, indirir, açar — ADIM 2/A1/A3 otomatik ATLANIR.

Cache `FAST` değerine göre ayrı tutulur (`fast40` vs `full`) çünkü FAST modda dataset
farklı (küçük stratified örneklem) içeriyor.</cell id="placeholder">

In [ ]:
import os, subprocess, time

CACHE_TAG = f"fast{FAST_N}" if FAST else "full"
CACHE_TAR = f"{DRIVE_RUNS}/nnUNet_preprocess_cache_801_{CACHE_TAG}.tar"
RESTORED = False

if os.path.isfile(CACHE_TAR):
    print(f"Drive'da hazir cache bulundu -> {CACHE_TAR}\nindiriliyor + aciliyor...")
    t0 = time.time()
    subprocess.run(["cp", CACHE_TAR, "/content/cache.tar"], check=True)
    subprocess.run(["tar", "-xf", "/content/cache.tar", "-C", WORK], check=True)
    os.remove("/content/cache.tar")
    RESTORED = (os.path.isdir(DST) and
                os.path.isdir(os.path.join(os.environ["nnUNet_preprocessed"], "Dataset801_IAC_LR")))
    print(f"cache acildi ({time.time() - t0:.0f}s) | ADIM 2 + A1(preprocess) + A3(split kurulumu) ATLANACAK | restored={RESTORED}")
else:
    print("Drive'da cache yok -> bu oturumda ADIM 2 + A1 + A3 calisacak, "
          "sonunda otomatik olarak Drive'a cache'lenecek (bir dahaki koparsa tekrar beklemezsin).")


# ADIM 2 — 3 sınıflı TRAINING dataset kur (SADECE P+F, S dışlanır)
IAC id'leri `dataset.json`'dan **isimle** çözülür. Kaynak veri RPI olmayan bir yönelimde
(ör. LPS) gelirse affine'in gerçek yön kosinüslerine göre RPI'ye çevrilir — fiziksel
sol/sağ korunur. `--subset PF`: S bu klasöre **hiç girmez** (nnU-Net'in otomatik 5-fold'u
her şeyi eğitimde kullanır, S'i ayırmayı bilmez — bu yüzden S'i baştan hiç koymuyoruz).
`FAST` iken `--ids-file outputs/fast_ids.json` ile ADIM 1'deki AYNI vaka listesi kullanılır.

In [ ]:
import subprocess
IDS_ARG = ["--ids-file", "outputs/fast_ids.json"] if FAST else []
if not RESTORED:
    subprocess.run(["python", "data/prepare_iac_dataset.py", "--src", TF3_SRC, "--subset", "PF",
                     *IDS_ARG, "--dst", DST, "--copy-images", "--workers", "4"], check=True)
else:
    print("cache'den geldi, ADIM 2 atlaniyor")
print("imagesTr:", len(os.listdir(IMAGES)), "| labelsTr:", len(os.listdir(LABELS)))

---
# TRACK A — nnU-Net v2 baseline

## A1. Plan & preprocess (dataset bütünlüğünü de doğrular)

In [ ]:
import subprocess
if not RESTORED:
    subprocess.run(["nnUNetv2_plan_and_preprocess", "-d", "801", "--verify_dataset_integrity"], check=True)
else:
    print("cache'den geldi, plan_and_preprocess atlaniyor")

## A2. Özel trainer'ları nnU-Net paketine kur (L/R için mirroring KAPALI)
Sol/sağ mirroring 1↔2 sınıflarını takas eder — domain kuralı gereği kapalı.

In [ ]:
import shutil, nnunetv2
_da = os.path.join(os.path.dirname(nnunetv2.__file__),
                   "training", "nnUNetTrainer", "variants", "data_augmentation")
shutil.copy("nnunet/nnUNetTrainerIAC.py", _da)
shutil.copy("nnunet/trainer_no_lr_mirror.py", _da)
print("trainer'lar kuruldu ->", _da)

## A3. Bizim split'i nnU-Net'in KENDİ split dosyasına kur — KRİTİK, leakage'i önler
nnU-Net'in kendi otomatik (rastgele) 5-fold split'i, `configs/splits.json`'daki P/F
stratified split'ten FARKLIDIR. Bu hücre olmadan `predict_oof.py`'ın "held-out" sandığı
vakalar gerçekte o modelin eğitiminde görülmüş olabilir — flow'un x0 prior'ı leakage'lı
olur. Bu, ADIM 1'deki `$SPLITS`'i (FAST'te fast, değilse tam) nnU-Net'e kurar.

In [ ]:
import subprocess
if not RESTORED:
    subprocess.run(["python", "nnunet/write_nnunet_splits.py", "--dataset-name", "Dataset801_IAC_LR",
                     "--splits", SPLITS], check=True)
else:
    print("cache'den geldi, split kurulumu atlaniyor (splits_final.json cache icinde zaten var)")

In [ ]:
import subprocess, time

if not RESTORED:
    print("cache olusturuluyor (bir sonraki oturumda ADIM 2 + A1 + A3'u atlamak icin)...")
    t0 = time.time()
    raw_rel  = os.path.relpath(DST, WORK)
    prep_rel = os.path.relpath(os.path.join(os.environ["nnUNet_preprocessed"], "Dataset801_IAC_LR"), WORK)
    subprocess.run(["tar", "-cf", "/content/cache.tar", "-C", WORK, raw_rel, prep_rel], check=True)
    subprocess.run(["cp", "/content/cache.tar", CACHE_TAR], check=True)
    os.remove("/content/cache.tar")
    RESTORED = True   # ayni oturumda bu hucre yanlislikla tekrar calisirsa ikinci kez yazmasin
    print(f"cache Drive'a yazildi ({time.time() - t0:.0f}s) -> {CACHE_TAR}")
else:
    print("bu oturumda zaten cache'den geldi, tekrar yazilmiyor")

## A4. 5 fold'un HEPSİNİ eğit — leakage-free OOF için ŞART
Flow'un başlangıç noktası (x0), her vakanın **onu eğitiminde görmemiş** bir modelce
tahminidir — bu artık A3'teki split kurulumu sayesinde gerçekten doğru. **En uzun adım budur.**

- `subprocess.run(check=True)`: bir fold gerçekten başarısız olursa burada durur —
  eskisi gibi sessizce sonraki fold'a geçip "bitti" görünümü vermez.
- `--c` sadece o fold'un GERÇEK bir `checkpoint_latest.pth`'i varsa eklenir (ilk
  çalıştırmada yok, koşulsuz `--c` bazı nnU-Net sürümlerinde hataya yol açabilir).
- Oturum koparsa: bu hücreyi **tekrar çalıştır** — kaldığı fold/epoch'tan devam eder.
- Checkpoint'ler Drive'da (`nnUNet_results`), kaybolmaz.

In [ ]:
import subprocess
FOLDS = [0, 1, 2, 3, 4]     # tam OOF icin hepsi; hizli bakis icin [0] (ama o zaman OOF/flow tam olmaz)
results_root = os.path.join(os.environ["nnUNet_results"], "Dataset801_IAC_LR",
                             f"{TRAINER}__nnUNetPlans__3d_fullres")
for f in FOLDS:
    fold_dir = os.path.join(results_root, f"fold_{f}")
    final_ck = os.path.join(fold_dir, "checkpoint_final.pth")
    latest_ck = os.path.join(fold_dir, "checkpoint_latest.pth")
    if os.path.isfile(final_ck):
        print(f"fold {f}: zaten bitmis, atlaniyor"); continue
    cmd = ["nnUNetv2_train", "801", "3d_fullres", str(f), "-tr", TRAINER]
    if os.path.isfile(latest_ck):
        cmd.append("--c")
        print(f"=== fold {f}: checkpoint_latest bulundu, devam ediliyor ===")
    else:
        print(f"=== fold {f}: sifirdan basliyor ===")
    subprocess.run(cmd, check=True)   # hata varsa burada durur (sessizce yutulmaz)
print("[nnU-Net] tum foldlar tamam.")

# ADIM 4 — leakage-free Out-Of-Fold tahminler → `outputs/oof_probs`
Her fold, kendi validation vakalarını tahmin eder (A3'te kurulan split sayesinde bu
vakalar o modelin eğitiminde GERÇEKTEN görülmemiştir). Flow'un prior'ı budur.

In [ ]:
!python nnunet/predict_oof.py --dataset 801 --config 3d_fullres \
    --trainer $TRAINER --splits "$SPLITS" \
    --images "$IMAGES" --out outputs/oof_probs
print("oof npz:", len(os.listdir("outputs/oof_probs")))

# ADIM 5 — SDF cache'leri (flow'un x1 hedefi ve x0 başlangıcı)
`compute_gt_sdf` = GT'den milimetre SDF (x1 hedefi). `compute_coarse_sdf` = OOF tahminden
SDF (x0 başlangıç, leakage-free). Bir kez hesaplanır, her epoch yeniden hesaplanmaz.

In [ ]:
!python data/compute_gt_sdf.py --labels "$LABELS" --out outputs/gt_sdf --clip-mm 10
!python data/compute_coarse_sdf.py --oof outputs/oof_probs --ref-labels "$LABELS" \
    --out outputs/coarse_sdf --prob-thresh 0.5 --clip-mm 10
print("gt_sdf:", len(os.listdir("outputs/gt_sdf")), "| coarse_sdf:", len(os.listdir("outputs/coarse_sdf")))

---
# TRACK B — residual flow refinement

# ADIM 6 — flow eğitimi (fold 0)
Checkpoint'ler Drive'a yazılır (`--out`), `--resume` ile kopmadan sonra devam eder.
**Bu SAME hücreyi tekrar çalıştırınca** kaldığı yerden sürer. T4'te VRAM için patch=96
büyük gelirse `configs/flow.yaml` içinde `patch: 64` yap. Diğer fold'lar için `--fold 1..4`.

In [ ]:
FLOW_OUT = f"{DRIVE_RUNS}/flow_fold0"
!python flow/train.py --config configs/flow.yaml --splits "$SPLITS" --fold 0 --resume \
    --images "$IMAGES" --labels "$LABELS" \
    --gt-sdf outputs/gt_sdf --coarse-sdf outputs/coarse_sdf \
    --out "$FLOW_OUT"
print("en iyi checkpoint ->", f"{FLOW_OUT}/best.pt")

# ADIM 7 (opsiyonel) — development üzerinde CV metrikleri
Bir tahmin klasörünü GT'ye karşı skorlar (Dice, HD95, clDice, NSD + topoloji).

```bash
# ornek: fold 0 val vakalarini baseline ile tahmin et, sonra skorla
# nnUNetv2_predict -i <val_imgs> -o outputs/preds_A -d 801 -c 3d_fullres -f 0 -tr $TRAINER --disable_tta
# python evaluation/evaluate_cv.py --pred outputs/preds_A --gt "$LABELS" --out outputs/cv_metrics.csv
```

---
# SON ADIM (bir kez, en sonda) — external S seti üzerinde değerlendirme

**S bu ana kadar hiçbir eğitim/preprocessing adımına girmedi.** Şimdi ayrı bir klasöre
hazırlanıp, eğitilmiş 5-fold modelle (ensemble) tahmin edilip skorlanıyor. `evaluate_external.py`
her skorlanan vaka `configs/splits.json`'ın `external_test` listesinde mi diye ayrıca doğrular
— değilse REFUSE eder (yanlışlıkla development vakası karışmasın diye). **Bunu yalnızca en
sonda, bir kez çalıştır** — sonucu görüp modele geri dönüp ayarlamamak için.

## Notlar
- **Sıra önemli:** split→dataset(PF)→plan/preprocess→split kurulumu(A3)→train→OOF→SDF→flow.
  A3 (split kurulumu) train'den ÖNCE olmalı — sonradan çalıştırmak zaten eğitilmiş fold'ları
  düzeltmez, yeniden eğitmek gerekir.
- **En pahalı kısım nnU-Net 5-fold.** Bütçe azsa `FAST=True` ile tüm zinciri bir kez
  uçtan uca gör; sayılar zayıf ama boru hattının **doğru** çalıştığını kanıtlar. Sonra `FAST=False`.
- **Her şey devam ettirilebilir:** nnU-Net `--c` (varsa), flow `--resume`, checkpoint'ler Drive'da.
- **Colab sık kopuyorsa (1.5. hücre):** ADIM 2 + A1 (preprocess) + A3 (split kurulumu) bir kez
  tamamlanınca Drive'a tek bir tar olarak cache'lenir; sonraki her oturum bunu bulup açar ve
  bu üç adımı atlar. `FAST_N`'i değiştirirsen (fast cache boyutunu), eski `fast{N}` tar'ı silmen
  gerekir — yoksa eski örneklemi geri yükler.
- **External S seti** yalnızca en sonda, bir kez, ayrı klasörde: SON ADIM hücresi.

## Notlar
- **Sıra önemli:** split→dataset(PF)→plan/preprocess→split kurulumu(A3)→train→OOF→SDF→flow.
  A3 (split kurulumu) train'den ÖNCE olmalı — sonradan çalıştırmak zaten eğitilmiş fold'ları
  düzeltmez, yeniden eğitmek gerekir.
- **En pahalı kısım nnU-Net 5-fold.** Bütçe azsa `FAST=True` ile tüm zinciri bir kez
  uçtan uca gör; sayılar zayıf ama boru hattının **doğru** çalıştığını kanıtlar. Sonra `FAST=False`.
- **Her şey devam ettirilebilir:** nnU-Net `--c` (varsa), flow `--resume`, checkpoint'ler Drive'da.
- **External S seti** yalnızca en sonda, bir kez, ayrı klasörde: SON ADIM hücresi.